# 2. Questions with fixed answers

All figures are saved to `outputs/figures/` (PNG, 300 dpi, and vector PDF). Percentages are shares of the respondents who answered the respective question; `n` is printed above each figure.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sys.path.append(str(Path.cwd().parent))
from src import plots as P
from src import schema as S
from src.data import (constraint_coverage, cooccurrence, crosstab_ordered, load_clean, multi_select_counts,
                      ordered_counts, ordinal_scores, select_population, trim_edges, yes_matrix)
from src.plots import save_figure

PROCESSED_PATH = Path("../data/processed/survey_clean.csv")
INCLUDE_PARTIAL = False  # True: every respondent who answered a question (n differs per figure)
P.SHOW_TITLES = True     # False: no titles (use the LaTeX caption instead)

df_all, sc = load_clean(PROCESSED_PATH)
df = select_population(df_all, sc, include_partial=INCLUDE_PARTIAL)
ROLE_COLORS = dict(zip(S.ROLE_GROUPS, P.GROUP_PALETTE))
print(f"{len(df)} respondents ({'all with answers' if INCLUDE_PARTIAL else 'completed surveys only'})")

## A. Sample

In [ ]:
counts = ordered_counts(df[sc.education].map(S.education_label), S.EDUCATION_ORDER, trim=True)
fig = P.bar_chart(counts, "Highest level of formal education", colors=P.sequential(len(counts)), wrap=26)
save_figure(fig, "sample_education")

In [ ]:
experience = pd.DataFrame({
    "Coding experience (total)": ordered_counts(df[sc.years_coding], S.YEARS_ORDER),
    "Professional experience": ordered_counts(df[sc.years_pro], S.YEARS_ORDER),
})
experience = trim_edges(experience)
experience.index = [S.years_label(i) for i in experience.index]
fig = P.grouped_bars(experience, [P.NAVY, P.ORANGE], "Programming and professional experience")
fig.axes[0].set_xlabel("Years")
save_figure(fig, "sample_experience")

In [ ]:
roles = pd.DataFrame({"role": df[sc.role].map(S.role_label), "group": df["role_group"]}).dropna()
counts = roles["role"].value_counts()
group_of = roles.drop_duplicates("role").set_index("role")["group"]
fig = P.bar_chart(counts, "Current role", n=len(roles), colors=[ROLE_COLORS[group_of[r]] for r in counts.index],
                  wrap=34)
P.add_legend(fig.axes[0], {g: ROLE_COLORS[g] for g in S.ROLE_GROUPS if g in set(roles["group"])})
save_figure(fig, "sample_role")

In [ ]:
languages = df[sc.languages].dropna().str.split(";").explode().str.strip().value_counts()
fig = P.bar_chart(languages, "Main programming language", n=int(df[sc.languages].notna().sum()), colors=P.TEAL)
save_figure(fig, "sample_language")

## B. Learning

In [ ]:
counts, n = multi_select_counts(df, sc.learn)
fig = P.lollipop_chart(counts, n, "How respondents learned to code in the past year", colors=P.TEAL)
save_figure(fig, "learning_sources")

In [ ]:
counts = ordered_counts(df[sc.ai_learning].map(S.ai_learning_label), S.AI_LEARNING_ORDER)
fig = P.bar_chart(counts, "Learning activities in the past year", wrap=32,
                  colors=[P.NAVY, "#7FA3C7", P.ORANGE, "#F0C089", P.GREY])
save_figure(fig, "learning_ai_at_work")

## C. Artificial intelligence

In [ ]:
counts = ordered_counts(df[sc.ai_use], S.AI_USE_ORDER, labels=S.AI_USE_LABELS)
fig = P.bar_chart(counts, "Use of AI tools in the development process", colors=P.sequential(len(counts))[::-1])
save_figure(fig, "ai_usage")

In [ ]:
stance = ordered_counts(df[sc.ai_stance], S.AI_STANCE_ORDER).to_frame("All respondents").T
fig = P.diverging_bars(stance, P.STANCE_COLORS, n_left=2, n_center=2,
                       title="Stance on using AI tools in the development workflow")
save_figure(fig, "ai_stance")

In [ ]:
counts = ordered_counts(df[sc.ai_threat], S.AI_THREAT_ORDER)
fig = P.donut_chart(counts, ["#1F6FA8", "#CFCFCF", "#B2501A"], "Is AI a threat to your current job?")
save_figure(fig, "ai_threat")

### AI stance in subgroups

In [ ]:
tab = crosstab_ordered(df, "ai_use_group", sc.ai_stance, S.AI_USE_GROUPS, S.AI_STANCE_ORDER)
fig = P.diverging_bars(tab, P.STANCE_COLORS, 2, 2, "AI stance by frequency of AI tool use")
save_figure(fig, "ai_stance_by_usage")

In [ ]:
tab = crosstab_ordered(df, "role_group", sc.ai_stance, S.ROLE_GROUPS, S.AI_STANCE_ORDER)
fig = P.diverging_bars(tab, P.STANCE_COLORS, 2, 2, "AI stance by role group")
save_figure(fig, "ai_stance_by_role")

In [ ]:
tab = crosstab_ordered(df, "pro_exp_group", sc.ai_stance, S.PRO_EXP_GROUPS, S.AI_STANCE_ORDER)
fig = P.diverging_bars(tab, P.STANCE_COLORS, 2, 2, "AI stance by professional experience")
save_figure(fig, "ai_stance_by_experience")

In [ ]:
tab = crosstab_ordered(df, "ai_stance_group", sc.ai_threat, S.AI_STANCE_GROUPS, S.AI_THREAT_ORDER)
fig = P.stacked_bars(tab, ["#1F6FA8", "#CFCFCF", "#B2501A"], "Perceived job threat by AI stance")
save_figure(fig, "ai_threat_by_stance")

## D. Non-functional requirements and project constraints

Both questions are multi-select; percentages are shares of respondents who selected an option.

In [ ]:
counts, n = multi_select_counts(df, sc.nfr)
fig = P.lollipop_chart(counts, n, "Non-functional requirements with the greatest influence (up to three)")
save_figure(fig, "nfr_ranking")
print("Selections per respondent:", yes_matrix(df, sc.nfr).sum(axis=1).describe().round(2).to_dict())

In [ ]:
nfr = yes_matrix(df, sc.nfr).drop(columns="Don't know")
fig = P.heatmap(cooccurrence(nfr), "Non-functional requirements selected together", mask_upper=True,
                cbar_label="Respondents", n=len(nfr))
save_figure(fig, "nfr_cooccurrence")

In [ ]:
counts, n = multi_select_counts(df, sc.constraints)
fig = P.lollipop_chart(counts, n, "Constraints with the greatest influence on development decisions", colors=P.ORANGE)
save_figure(fig, "constraints_ranking")

In [ ]:
constraints = yes_matrix(df, sc.constraints).drop(columns="Don't know")
fig = P.heatmap(cooccurrence(constraints), "Constraints selected together", mask_upper=True,
                cbar_label="Respondents", cmap=sns.light_palette(P.ORANGE, as_cmap=True), n=len(constraints))
save_figure(fig, "constraints_cooccurrence")

In [ ]:
mc, mn = constraints, nfr
common = mc.index.intersection(mn.index)
assoc = mc.loc[common].astype(int).T @ mn.loc[common].astype(int)
fig = P.heatmap(assoc, "Selected constraints vs. selected non-functional requirements", n=len(common),
                cbar_label="Respondents selecting both", row_wrap=30, col_wrap=14)
save_figure(fig, "constraints_vs_nfr")

## E. Development process and outlook

In [ ]:
counts = ordered_counts(df[sc.flexibility], S.FLEXIBILITY_ORDER, labels=S.flexibility_label, trim=True)
levels = [c for c in counts.index if c != "I don't know"]
shade = dict(zip(levels, P.sequential(len(levels))))
fig = P.bar_chart(counts, "Flexibility to choose or adapt the development process",
                  colors=[shade.get(c, P.GREY) for c in counts.index])
save_figure(fig, "process_flexibility")

In [ ]:
cert = df[sc.constraints["Certification / regulation"]].map(
    {"Yes": "Certification / regulation selected", "No": "Not selected"})
tab = pd.crosstab(cert, df["flex_group"]).reindex(
    index=["Certification / regulation selected", "Not selected"], columns=S.FLEXIBILITY_GROUPS, fill_value=0)
fig = P.stacked_bars(tab, P.sequential(3), "Process flexibility by certification constraint")
save_figure(fig, "flexibility_by_certification")

In [ ]:
tab = crosstab_ordered(df, "role_group", "flex_group", S.ROLE_GROUPS, S.FLEXIBILITY_GROUPS)
fig = P.stacked_bars(tab, P.sequential(3), "Process flexibility by role group")
save_figure(fig, "flexibility_by_role")

In [ ]:
overall = ordered_counts(df[sc.similar_choice], S.SIMILAR_ORDER, labels=S.SIMILAR_LABELS).to_frame("All respondents").T
fig = P.diverging_bars(overall, P.DIVERGING5, n_left=2, n_center=1,
                       title="Will space-sector software engineering become more similar to other domains?")
save_figure(fig, "similar_or_different")

In [ ]:
for col, order, name, title in [
    ("ai_stance_group", S.AI_STANCE_GROUPS, "ai_stance", "by AI stance"),
    ("role_group", S.ROLE_GROUPS, "role", "by role group"),
]:
    tab = crosstab_ordered(df, col, sc.similar_choice, order, S.SIMILAR_ORDER)
    tab.columns = [S.SIMILAR_LABELS[c] for c in tab.columns]
    fig = P.diverging_bars(tab, P.DIVERGING5, 2, 1, f"Expected convergence with other domains, {title}")
    save_figure(fig, f"similar_or_different_by_{name}")

## F. Exploratory associations

Spearman rank correlations between the ordinal variables (pairwise complete cases). With about 25 respondents only $|\rho| \gtrsim 0.4$ is distinguishable from noise, so this is a pointer for further analysis, not a test result.

In [ ]:
scores = pd.DataFrame({
    "Professional experience": ordinal_scores(df[sc.years_pro], S.YEARS_ORDER),
    "Coding experience": ordinal_scores(df[sc.years_coding], S.YEARS_ORDER),
    "AI tool usage frequency": 4 - ordinal_scores(df[sc.ai_use], S.AI_USE_ORDER),
    "Favorable towards AI": ordinal_scores(df[sc.ai_stance], ["Very unfavorable", "Unfavorable", "Indifferent",
                                                              "Favorable", "Very favorable"]),
    "Process flexibility": ordinal_scores(df[sc.flexibility], S.FLEXIBILITY_ORDER[:-1]),
    "Expects convergence": ordinal_scores(df[sc.similar_choice], S.SIMILAR_ORDER),
})
rho = scores.corr(method="spearman").iloc[1:, :-1]  # drop the trivial diagonal
fig = P.heatmap(rho, "Spearman correlations between ordinal survey variables", fmt=".2f", hide_zero=False,
                mask_upper=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar_label="Spearman ρ",
                n=len(scores))
save_figure(fig, "ordinal_correlations")